# 🛣️ ROADGUARD AI — HUẤN LUYỆN YOLOv11 TRÊN ~15,000 ẢNH (TỐI ƯU COLAB FREE)
### Nhận Diện Ổ Gà (Pothole) + Vết Nứt (Crack) + Mẫu Âm Tính Chống Báo Động Giả
---
> **Dataset:** `unified_road_dataset_15k.zip` (~15,000 ảnh + 15,000 file nhãn `.txt`)  
> **Thành phần:** 3,940 pothole + 5,000 crack + 6,060 negative (background)  
> **2 Classes:** `pothole` (0), `crack` (1)  
> **Phần cứng:** Google Colab GPU Tesla T4 (Miễn phí)  
> **Đầu ra:** File trọng số xuất xưởng `roadguard_best_15k.pt`
>
> ⏱️ **Dự kiến:** ~30–50 phút / 50 epoch trên T4 — rất an toàn trong session Colab Free

## 📌 Bước 1: Kiểm Tra GPU T4 & Cài Đặt Thư Viện

In [ ]:
# 1. Kiểm tra trạng thái GPU
!nvidia-smi

import torch
print("=" * 65)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Dung lượng:{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU")
print("=" * 65)

# 2. Cài đặt Ultralytics mới nhất
!pip install -q ultralytics

## 📥 Bước 2: Tải & Giải Nén File `unified_road_dataset_15k.zip`
---
> **Lưu ý:** File zip nặng **~279 MB**. Có 2 cách để tải lên:
>
> **Cách 1 (Kéo thả):** Kéo thả file `unified_road_dataset_15k.zip` vào cột Files bên trái của Colab,  
> hãy đợi **vòng tròn upload xoay xong 100%** rồi mới chạy ô bên dưới.
>
> **Cách 2 (Google Drive - Khuyên dùng):** Upload file lên Google Drive trước, rồi bỏ dấu `#` ở 3 dòng Mount Drive bên dưới để đọc siêu tốc chỉ mất ~3 giây.

In [ ]:
# (Tùy chọn) Bỏ dấu '#' ở 3 dòng dưới nếu bạn upload file zip lên Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/unified_road_dataset_15k.zip" /content/ || echo 'Drive copy fallback'

In [ ]:
import os, glob, zipfile
from pathlib import Path

dataset_dir = "/content/dataset"
os.makedirs(dataset_dir, exist_ok=True)

# 1. Quét tìm file zip dataset trong Colab
zip_files = glob.glob("/content/*.zip") + glob.glob("*.zip")
print(f"🔍 Tìm thấy các file zip: {zip_files}")

if not zip_files:
    raise FileNotFoundError("❌ CHƯA THẤY FILE ZIP! Hãy kéo thả file 'unified_road_dataset_15k.zip' vào cột Files bên trái Colab rồi chạy lại ô này.")

# Ưu tiên chọn file 15k
chosen_zip = next((z for z in zip_files if "15k" in z.lower()), zip_files[0])
size_mb = os.path.getsize(chosen_zip) / (1024 * 1024)
EXPECTED_MB = 279  # dung lượng chuẩn của file 15k
print("=" * 65)
print(f"📦 File đang chọn: {chosen_zip}")
print(f"📊 Dung lượng hiện tại: {size_mb:.2f} MB / {EXPECTED_MB} MB chuẩn")

if size_mb < EXPECTED_MB * 0.90:
    print("⚠️ CẢNH BÁO: File chưa upload xong!")
    print(f"   Mới tải được {size_mb:.2f} MB / {EXPECTED_MB} MB.")
    print("   Hãy đợi vòng tròn upload xoay xong 100% rồi chạy lại ô này.")
    raise RuntimeError(f"Upload chưa xong: {size_mb:.1f}MB / {EXPECTED_MB}MB")

# 2. Giải nén
print(f"🚀 Đang giải nén: {chosen_zip} → {dataset_dir}...")
res = os.system(f"unzip -q -o '{chosen_zip}' -d '{dataset_dir}'")
if res != 0:
    print("ℹ️ Thử giải nén dự phòng bằng Python zipfile...")
    with zipfile.ZipFile(chosen_zip, 'r') as z:
        z.extractall(dataset_dir)
print("🎉 GIẢI NÉN THÀNH CÔNG!")

# 3. Định vị và ghi cố định file data.yaml
yaml_candidates = glob.glob(f"{dataset_dir}/**/data.yaml", recursive=True)
yaml_path = os.path.abspath(yaml_candidates[0]) if yaml_candidates else f"{dataset_dir}/data.yaml"
root_extracted = str(Path(yaml_path).parent).replace('\\', '/')

fixed_yaml = f"""path: {root_extracted}
train: train/images
val:   valid/images
test:  test/images

nc: 2
names: ['pothole', 'crack']
"""
with open(yaml_path, 'w') as f:
    f.write(fixed_yaml)

print("=" * 65)
print(f"✅ data.yaml đã cấu hình tại: {yaml_path}")
print(fixed_yaml)

# 4. Thống kê ảnh
train_cnt = len(glob.glob(f"{root_extracted}/train/images/*.*"))
val_cnt   = len(glob.glob(f"{root_extracted}/valid/images/*.*"))
test_cnt  = len(glob.glob(f"{root_extracted}/test/images/*.*"))
print(f"📊 Train={train_cnt:,} | Valid={val_cnt:,} | Test={test_cnt:,} | TỔNG={train_cnt+val_cnt+test_cnt:,} ảnh")
print("=" * 65)

## 🔍 Bước 3: Trực Quan Hóa Mẫu Dataset (Kiểm Tra Trước Khi Train)

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plt.style.use('dark_background')

def show_yolo_samples(img_dir, lbl_dir, title, n=6):
    imgs = glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png")
    if not imgs:
        print(f"Không tìm thấy ảnh trong {img_dir}")
        return
    samples = random.sample(imgs, min(n, len(imgs)))
    class_colors = {0: '#e74c3c', 1: '#00bcd4'}  # đỏ = pothole, xanh = crack
    class_names  = {0: 'pothole', 1: 'crack'}

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title, fontsize=13, fontweight='bold', color='white')

    for ax, img_path in zip(axes.flat, samples):
        img = Image.open(img_path)
        W, H = img.size
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path), fontsize=7, color='#aaaaaa')
        ax.axis('off')

        lbl_path = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls = int(parts[0])
                        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                        x1 = (xc - bw/2) * W
                        y1 = (yc - bh/2) * H
                        color = class_colors.get(cls, 'white')
                        rect = patches.Rectangle((x1, y1), bw*W, bh*H,
                                                  linewidth=2, edgecolor=color, facecolor='none')
                        ax.add_patch(rect)
                        ax.text(x1, max(y1-4, 0), class_names.get(cls, str(cls)),
                                color=color, fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Hiển thị 6 ảnh mẫu tập Train
show_yolo_samples(
    f"{root_extracted}/train/images",
    f"{root_extracted}/train/labels",
    "🔍 Mẫu ảnh TRAIN — Ổ gà 🔴 | Vết nứt 🔵 | Negative (không có bbox)"
)

# Biểu đồ phân bổ nhãn
print("\n📊 Đang phân tích phân bổ nhãn tập Train...")
counts = {'pothole': 0, 'crack': 0, 'negative': 0}
for lbl_f in glob.glob(f"{root_extracted}/train/labels/*.txt"):
    with open(lbl_f) as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        counts['negative'] += 1
    else:
        for l in lines:
            cls = int(l.split()[0])
            if cls == 0: counts['pothole'] += 1
            elif cls == 1: counts['crack'] += 1

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.keys(), counts.values(),
              color=['#e74c3c', '#00bcd4', '#7f8c8d'], edgecolor='white', linewidth=0.8)
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')
ax.set_title('Phân Bổ Nhãn — Tập Train (15k Dataset)', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
for bar, (k, v) in zip(bars, counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{v:,}', ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 Bước 4: Huấn Luyện Mô Hình YOLOv11 (2 Classes: Pothole & Crack)
---
> **Cấu hình tối ưu cho Colab Free (15k ảnh):**
> | Tham số | Giá trị | Lý do |
> |---------|---------|-------|
> | `epochs` | 50 | Đủ hội tụ, không overfit |
> | `batch` | 16 | Phù hợp VRAM 15GB T4 |
> | `workers` | 2 | Tránh overhead CPU Colab |
> | `cache` | False | Tránh OOM RAM 12GB |
> | `imgsz` | 640 | Khớp với Tiling Engine 640×640 |

In [ ]:
from ultralytics import YOLO

# Khởi tạo YOLOv11 Nano — tối ưu cho suy luận thời gian thực trên Drone/Edge AI
model = YOLO('yolo11n.pt')

print("🚀 BẮT ĐẦU HUẤN LUYỆN YOLOv11 TRÊN 15,000 ẢNH VỚI GPU T4...")
print("⏱️ Dự kiến: ~30–50 phút / 50 epoch")
print("=" * 65)

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    save=True,
    save_period=10,
    plots=True,
    name='roadguard_15k_yolo11'
)

print("=" * 65)
print("🎉 HUẤN LUYỆN HOÀN TẤT 50 EPOCHS!")
print(f"   mAP50   : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"   mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print("=" * 65)

## 📊 Bước 5: Trực Quan Hóa Kết Quả Huấn Luyện

In [ ]:
from IPython.display import Image as IPImage, display

run_dir = "runs/detect/roadguard_15k_yolo11"

# 1. Learning Curves (Loss + mAP50)
if os.path.exists(f"{run_dir}/results.png"):
    print("📈 Biểu đồ đường cong học tập (Loss & mAP50):")
    display(IPImage(f"{run_dir}/results.png"))

# 2. Confusion Matrix
if os.path.exists(f"{run_dir}/confusion_matrix.png"):
    print("🎯 Ma trận nhầm lẫn (Confusion Matrix):")
    display(IPImage(f"{run_dir}/confusion_matrix.png"))

# 3. Mẫu dự đoán trên tập validation
val_preds = glob.glob(f"{run_dir}/val_batch*_pred.jpg")
if val_preds:
    print("🔍 Mẫu ảnh dự đoán trên tập validation:")
    display(IPImage(val_preds[0]))

## 🔬 Bước 6: Kiểm Thử Trên Tập Test (Đánh Giá Cuối Cùng)

In [ ]:
from ultralytics import YOLO

best_model = YOLO(f"{run_dir}/weights/best.pt")

print("🔬 Đang chạy kiểm thử trên 751 ảnh tập Test...")
metrics = best_model.val(
    data=yaml_path,
    split='test',
    device=0
)

print("\n" + "=" * 65)
print("📋 KẾT QUẢ KIỂM THỬ TẬP TEST:")
print(f"   Precision : {metrics.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"   Recall    : {metrics.results_dict.get('metrics/recall(B)', 0):.4f}")
print(f"   mAP@0.50  : {metrics.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"   mAP@0.5:95: {metrics.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print("=" * 65)

## 💾 Bước 7: Xuất Xưởng & Tải `roadguard_best_15k.pt` Về Máy

In [ ]:
import shutil
from google.colab import files

src_weight      = f"{run_dir}/weights/best.pt"
out_weight_name = "roadguard_best_15k.pt"

if os.path.exists(src_weight):
    shutil.copy(src_weight, out_weight_name)
    print(f"✅ Đã xuất xưởng: {out_weight_name}")
    print("⬇️ Đang kích hoạt tải file về máy...")
    files.download(out_weight_name)
    print("=" * 65)
    print("👉 Sau khi tải xong, chép file vào:")
    print(f"   F:\\train AI đồ án\\weights\\trained\\{out_weight_name}")
    print("=" * 65)
else:
    print(f"❌ Không tìm thấy: {src_weight}")
    print("   Hãy kiểm tra Bước 4 đã chạy thành công chưa.")

## ☁️ (Tùy chọn) Bước 8: Sao Lưu Kết Quả Lên Google Drive

In [ ]:
# Bỏ dấu '#' ở các dòng dưới để sao lưu toàn bộ kết quả train lên Drive
# (Bao gồm: checkpoints, biểu đồ, confusion matrix, weights best + last)

# from google.colab import drive
# drive.mount('/content/drive')  # Bỏ qua nếu đã mount ở Bước 2

# backup_dir = "/content/drive/MyDrive/RoadGuard_AI_15k_Results"
# os.makedirs(backup_dir, exist_ok=True)
# !cp -r /content/runs/detect/roadguard_15k_yolo11 "{backup_dir}/"
# print(f"✅ Đã sao lưu kết quả train vào Drive: {backup_dir}")